# LIBRARIES

In [11]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import random
from queue import Queue, PriorityQueue
import scipy.stats as st
import numpy as np




# CLASSES

## MEASURES

In [13]:
class Measure:
    def __init__(self,Narr,Ndep,NAveraegUser,OldTimeEvent,AverageDelay):
        self.arr = Narr
        self.dep = Ndep
        self.ut = NAveraegUser
        self.oldT = OldTimeEvent
        self.delay = AverageDelay

## CLIENT

In [14]:
class Client:
    def __init__(self,type,arrival_time):
        self.type = type
        self.arrival_time = arrival_time

## SERVER

In [15]:
class Server(object):

    # constructor
    def __init__(self):

        # whether the server is idle or not
        self.idle = True

# FUNCTIONS

## ARRIVAL

In [16]:
def arrival(time, FES, queue):
    global users
    
    #print("Arrival no. ",data.arr+1," at time ",time," with ",users," users" )
    
    # cumulate statistics
    data.arr += 1
    data.ut += users*(time-data.oldT)
    data.oldT = time

    # sample the time until the next event
    inter_arrival = random.expovariate(lambd=1.0/ARRIVAL)
    
    # schedule the next arrival
    FES.put((time + inter_arrival, "arrival"))

    users += 1
    
    # create a record for the client
    client = Client(TYPE1,time)

    # insert the record in the queue
    queue.append(client)

    # if the server is idle start the service
    if users==1:
        
        # sample the service time
        service_time = random.expovariate(1.0/SERVICE)
        #service_time = 1 + random.uniform(0, SEVICE_TIME)

        # schedule when the client will finish the server
        FES.put((time + service_time, "departure"))

## DEPARTURES

In [17]:
def departure(time, FES, queue):
    global users

    #print("Departure no. ",data.dep+1," at time ",time," with ",users," users" )
        
    # cumulate statistics
    data.dep += 1
    data.ut += users*(time-data.oldT)
    data.oldT = time
    
    # get the first element from the queue
    client = queue.pop(0)
    
    # do whatever we need to do when clients go away
    
    data.delay += (time-client.arrival_time)
    users -= 1
    
    # see whether there are more clients to in the line
    if users >0:
        # sample the service time
        service_time = random.expovariate(1.0/SERVICE)

        # schedule when the client will finish the server
        FES.put((time + service_time, "departure"))


## CONFIDENCE INTERVALS

In [18]:

def confidence_interval(data, confidence=0.95):
    
    n = len(data)
    mean = np.mean(data)
    std_err = st.sem(data)
    h = std_err * st.t.ppf((1 + confidence) / 2, n - 1)

    return mean, mean - h, mean + h

# CONSTANT

In [22]:

SERVICE = 10.0 # SERVICE is the average service time; service rate = 1/SERVICE
ARRIVAL =[3 *i for i in range(1, 5)]  # ARRIVAL is the average inter-arrival time; arrival rate = 1/ARRIVAL
LOAD=[SERVICE/arrival for arrival in ARRIVAL] # This relationship holds for M/M/1

TYPE1 = 1 

SIM_TIME = 500000

arrivals=0
users=0
BusyServer=False # True: server is currently busy; False: server is currently idle

MM1=[]

# MAIN

In [23]:
random.seed(42)

data = Measure(0,0,0,0,0)

# the simulation time 
time = 0

# the list of events in the form: (time, type)
FES = PriorityQueue()


# schedule the first arrival at t=0
FES.put((0, "arrival"))

# simulate until the simulated time reaches a constant
while time < SIM_TIME:
    (time, event_type) = FES.get()

    if event_type == "arrival":
        arrival(time, FES, MM1)

    elif event_type == "departure":
        departure(time, FES, MM1)

# print output data
print("MEASUREMENTS \n\nNo. of users in the queue:",users,"\nNo. of arrivals =",
      data.arr,"- No. of departures =",data.dep)

print("Load: ",SERVICE/ARRIVAL)
print("\nArrival rate: ",data.arr/time," - Departure rate: ",data.dep/time)

print("\nAverage number of users: ",data.ut/time)

print("Average delay: ",data.delay/data.dep)
print("Actual queue size: ",len(MM1))

if len(MM1)>0:
    print("Arrival time of the last element in the queue:",MM1[len(MM1)-1].arrival_time)
    


TypeError: unsupported operand type(s) for /: 'float' and 'list'

In [ ]:

# Run 25 simulations for each scenario (ARRIVAL value) and collect results
num_simulations = 25
all_results = {}

for arrival_scenario in ARRIVAL:
    results = {
        'arrivals': [],
        'departures': [],
        'avg_users': [],
        'avg_delay': [],
        'arrival_rate': [],
        'departure_rate': []
    }
    
    for sim in range(num_simulations):
        # Reset random seed for reproducibility
        random.seed(42 + sim)
        
        # Reset global variables for this simulation
        users = 0
        data = Measure(0, 0, 0, 0, 0)
        MM1 = []
        
        # the simulation time 
        time = 0
        
        # the list of events in the form: (time, type)
        FES = PriorityQueue()
        
        # schedule the first arrival at t=0
        FES.put((0, "arrival"))
        
        # Temporarily override ARRIVAL for this simulation
        original_arrival = globals()['ARRIVAL']
        globals()['ARRIVAL'] = arrival_scenario
        
        # simulate until the simulated time reaches a constant
        while time < SIM_TIME:
            (time, event_type) = FES.get()
        
            if event_type == "arrival":
                arrival(time, FES, MM1)
        
            elif event_type == "departure":
                departure(time, FES, MM1)
        
        # Restore original ARRIVAL
        globals()['ARRIVAL'] = original_arrival
        
        # Collect results from this simulation
        results['arrivals'].append(data.arr)
        results['departures'].append(data.dep)
        results['avg_users'].append(data.ut / time)
        results['avg_delay'].append(data.delay / data.dep if data.dep > 0 else 0)
        results['arrival_rate'].append(data.arr / time)
        results['departure_rate'].append(data.dep / time)
    
    all_results[arrival_scenario] = results

# Print summary statistics for each scenario
for arrival_scenario in ARRIVAL:
    results = all_results[arrival_scenario]
    
    print("=" * 60)
    print(f"RESULTS FROM 25 SIMULATIONS - ARRIVAL = {arrival_scenario}")
    print("=" * 60)
    
    print("\nNUMBER OF ARRIVALS:")
    mean, lower, upper = confidence_interval(results['arrivals'])
    print(f"  Mean: {mean:.2f}, CI[95%]: [{lower:.2f}, {upper:.2f}]")
    
    print("\nNUMBER OF DEPARTURES:")
    mean, lower, upper = confidence_interval(results['departures'])
    print(f"  Mean: {mean:.2f}, CI[95%]: [{lower:.2f}, {upper:.2f}]")
    
    print("\nAVERAGE NUMBER OF USERS IN SYSTEM:")
    mean, lower, upper = confidence_interval(results['avg_users'])
    print(f"  Mean: {mean:.4f}, CI[95%]: [{lower:.4f}, {upper:.4f}]")
    
    print("\nAVERAGE DELAY (per customer):")
    mean, lower, upper = confidence_interval(results['avg_delay'])
    print(f"  Mean: {mean:.4f}, CI[95%]: [{lower:.4f}, {upper:.4f}]")
    
    print("\nARRIVAL RATE:")
    mean, lower, upper = confidence_interval(results['arrival_rate'])
    print(f"  Mean: {mean:.6f}, CI[95%]: [{lower:.6f}, {upper:.6f}]")
    
    print("\nDEPARTURE RATE:")
    mean, lower, upper = confidence_interval(results['departure_rate'])
    print(f"  Mean: {mean:.6f}, CI[95%]: [{lower:.6f}, {upper:.6f}]")
    
    print(f"\nLOAD FACTOR (λ/μ) = SERVICE/ARRIVAL = {SERVICE/arrival_scenario:.4f}")
    
    print()


TypeError: unsupported operand type(s) for /: 'float' and 'list'